# ETL Demonstration: Student Records

**Course Topic:** Data Integration and ETL  
**Tools:** Python, Pandas, Excel, SQLite  
**Learning Focus:** Extraction, Data Integration, Data Cleansing, Transformation, Validation, Data Mapping, Loading, Batch Processing, and a simple Real-Time Integration simulation.

### Scenario
A school receives student master data and enrollment data from separate sources. The objective is to clean, validate, transform, integrate, and load the data into a target database.


## 1. ETL Workflow

```text
students_raw + enrollment_raw
            ↓
         EXTRACT
            ↓
      DATA INTEGRATION
            ↓
      DATA CLEANSING
            ↓
       TRANSFORMATION
            ↓
        VALIDATION
            ↓
           LOAD
            ↓
      SQLite Database
```

We will intentionally work with imperfect data so that validation and cleansing can be demonstrated.


In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

EXCEL_FILE = Path("ETL_Student_Records_Demo.xlsx")

students = pd.read_excel(EXCEL_FILE, sheet_name="students_raw")
enrollment = pd.read_excel(EXCEL_FILE, sheet_name="enrollment_raw")

print("Students:")
display(students)

print("\nEnrollment:")
display(enrollment)


Students:


,student_id,name,course
0,S001,Juan Dela Cruz,BSIT
1,S002,Maria Santos,BSCS
2,S003,Pedro Reyes,BSIT
3,S004,Anna Cruz,BSIS
4,S005,John Doe,BSIT
5,S007,Mark Garcia,NaN



Enrollment:


,student_id,subject,units
0,S001,IT101,3
1,S002,IT101,3
2,S003,IT101,3
3,S003,IT101,3
4,S004,IT101,3
5,S006,IT101,3
6,S005,IT101,-3
7,S007,IT101,3


## 2. Extraction

**Extraction** retrieves data from source systems.

In this demonstration, the source is an Excel workbook with two independent data sources:

- `students_raw`
- `enrollment_raw`

In an actual ETL system, these could instead be CSV files, databases, APIs, cloud storage, or IoT sources.


## 3. Data Quality Inspection

Before transforming the data, inspect its structure and identify potential problems.


In [2]:
print("Students information:")
students.info()

print("\nEnrollment information:")
enrollment.info()

print("\nMissing values:")
display(students.isna().sum().to_frame("students_missing"))
display(enrollment.isna().sum().to_frame("enrollment_missing"))

print("\nDuplicate enrollment records:")
display(enrollment[enrollment.duplicated(keep=False)])


Students information:
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   student_id  6 non-null      str  
 1   name        6 non-null      str  
 2   course      5 non-null      str  
dtypes: str(3)
memory usage: 388.0 bytes

Enrollment information:
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   student_id  8 non-null      str  
 1   subject     8 non-null      str  
 2   units       8 non-null      int64
dtypes: int64(1), str(2)
memory usage: 396.0 bytes

Missing values:


,students_missing
student_id,0
name,0
course,1


,enrollment_missing
student_id,0
subject,0
units,0



Duplicate enrollment records:


,student_id,subject,units
2,S003,IT101,3
3,S003,IT101,3


## 4. Data Integration

Integrate the enrollment data with the student master data using `student_id`.

The `left` join keeps all enrollment records so that unmatched records such as `S006` can be detected.


In [3]:
integrated = enrollment.merge(
    students,
    on="student_id",
    how="left",
    indicator=True
)

display(integrated)


,student_id,subject,units,name,course,_merge
0,S001,IT101,3,Juan Dela Cruz,BSIT,both
1,S002,IT101,3,Maria Santos,BSCS,both
2,S003,IT101,3,Pedro Reyes,BSIT,both
3,S003,IT101,3,Pedro Reyes,BSIT,both
4,S004,IT101,3,Anna Cruz,BSIS,both
5,S006,IT101,3,NaN,NaN,left_only
6,S005,IT101,-3,John Doe,BSIT,both
7,S007,IT101,3,Mark Garcia,NaN,both


## 5. Detect Validation Problems

### Validation rules

1. Student must exist in the student master data.
2. Student name must not be missing.
3. Program/course should be available.
4. Units must be greater than zero.
5. Duplicate enrollment records are not allowed.


In [4]:
# Unknown students
unknown_students = integrated[integrated["_merge"] == "left_only"]
print("Unknown students:")
display(unknown_students)

# Invalid units
invalid_units = integrated[integrated["units"] <= 0]
print("Invalid units:")
display(invalid_units)

# Duplicate records
duplicates = integrated[integrated.duplicated(
    subset=["student_id", "subject", "units"],
    keep=False
)]
print("Duplicate enrollment records:")
display(duplicates)


Unknown students:


,student_id,subject,units,name,course,_merge
5,S006,IT101,3,NaN,NaN,left_only


Invalid units:


,student_id,subject,units,name,course,_merge
6,S005,IT101,-3,John Doe,BSIT,both


Duplicate enrollment records:


,student_id,subject,units,name,course,_merge
2,S003,IT101,3,Pedro Reyes,BSIT,both
3,S003,IT101,3,Pedro Reyes,BSIT,both


## 6. Data Cleansing

We will:

- Trim unnecessary whitespace from names.
- Remove duplicate enrollment records.
- Remove enrollment records for students that do not exist.
- Remove records with invalid units.
- Fill a missing program with `UNKNOWN` for demonstration purposes.

> In a production system, the business rule may instead require missing values to be rejected and sent to an error/quarantine table.


In [5]:
cleaned = integrated.copy()

# Trim whitespace
cleaned["name"] = cleaned["name"].astype("string").str.strip()

# Remove unmatched students
cleaned = cleaned[cleaned["_merge"] == "both"].copy()

# Remove invalid units
cleaned = cleaned[cleaned["units"] > 0].copy()

# Remove duplicates
cleaned = cleaned.drop_duplicates(
    subset=["student_id", "subject", "units"]
).copy()

# Handle missing course/program
cleaned["course"] = cleaned["course"].fillna("UNKNOWN")

display(cleaned)


,student_id,subject,units,name,course,_merge
0,S001,IT101,3,Juan Dela Cruz,BSIT,both
1,S002,IT101,3,Maria Santos,BSCS,both
2,S003,IT101,3,Pedro Reyes,BSIT,both
4,S004,IT101,3,Anna Cruz,BSIS,both
7,S007,IT101,3,Mark Garcia,UNKNOWN,both


## 7. Transformation

Transformation changes data into the structure and format required by the target system.

Transformations in this example:

- Rename fields.
- Standardize text to uppercase.
- Calculate `TotalHours = Units × 18`.


In [6]:
transformed = cleaned.copy()

transformed["name"] = transformed["name"].str.strip()
transformed["course"] = transformed["course"].str.upper()
transformed["subject"] = transformed["subject"].str.upper()
transformed["total_hours"] = transformed["units"] * 18

transformed = transformed.rename(columns={
    "student_id": "StudentID",
    "name": "StudentName",
    "course": "Program",
    "subject": "SubjectCode",
    "units": "Units",
    "total_hours": "TotalHours"
})

transformed = transformed[
    ["StudentID", "StudentName", "Program",
     "SubjectCode", "Units", "TotalHours"]
]

display(transformed)


,StudentID,StudentName,Program,SubjectCode,Units,TotalHours
0,S001,Juan Dela Cruz,BSIT,IT101,3,54
1,S002,Maria Santos,BSCS,IT101,3,54
2,S003,Pedro Reyes,BSIT,IT101,3,54
4,S004,Anna Cruz,BSIS,IT101,3,54
7,S007,Mark Garcia,UNKNOWN,IT101,3,54


## 8. Data Mapping

| Source Field | Target Field | Rule |
|---|---|---|
| student_id | StudentID | Direct mapping |
| name | StudentName | Trim whitespace |
| course | Program | Uppercase |
| subject | SubjectCode | Uppercase |
| units | Units | Must be > 0 |
| units | TotalHours | Units × 18 |

This mapping specification documents how source data becomes target data.


## 9. Final Validation

Validate the transformed dataset before loading it into the target database.


In [7]:
assert transformed["StudentID"].notna().all()
assert transformed["StudentName"].notna().all()
assert transformed["Program"].notna().all()
assert (transformed["Units"] > 0).all()
assert not transformed.duplicated(
    subset=["StudentID", "SubjectCode", "Units"]
).any()

print("VALIDATION PASSED: Data is ready for loading.")


VALIDATION PASSED: Data is ready for loading.


## 10. Loading into SQLite

The target system will be a SQLite database.

This represents the **Load** stage of ETL.


In [8]:
DB_FILE = "school_etl.db"

connection = sqlite3.connect(DB_FILE)

transformed.to_sql(
    "student_enrollment",
    connection,
    if_exists="replace",
    index=False
)

connection.close()

print(f"Data loaded successfully into {DB_FILE}")


Data loaded successfully into school_etl.db


## 11. Verify the Loaded Data

Query the target database to confirm that the ETL pipeline produced the expected result.


In [9]:
connection = sqlite3.connect(DB_FILE)

result = pd.read_sql(
    "SELECT * FROM student_enrollment",
    connection
)

display(result)

connection.close()


,StudentID,StudentName,Program,SubjectCode,Units,TotalHours
0,S001,Juan Dela Cruz,BSIT,IT101,3,54
1,S002,Maria Santos,BSCS,IT101,3,54
2,S003,Pedro Reyes,BSIT,IT101,3,54
3,S004,Anna Cruz,BSIS,IT101,3,54
4,S007,Mark Garcia,UNKNOWN,IT101,3,54


## 12. Batch Processing Simulation

**Batch processing** handles data in groups at scheduled intervals.

Example:

```text
Every night at 11:00 PM
        ↓
Extract all new enrollment records
        ↓
Clean + Transform + Validate
        ↓
Load into database
```

The current notebook itself is a batch-style ETL demonstration because the complete dataset is processed as a group.


In [10]:
# Simple batch simulation
batch = enrollment.copy()

print(f"Batch received: {len(batch)} raw records")
print("Processing batch...")
print("Extract → Integrate → Clean → Transform → Validate → Load")
print("Batch processing completed.")


Batch received: 8 raw records
Processing batch...
Extract → Integrate → Clean → Transform → Validate → Load
Batch processing completed.


## 13. Real-Time Data Integration Simulation

Real-time integration processes data as it arrives rather than waiting for a scheduled batch.

Here, we simulate a new enrollment event arriving.


In [11]:
new_enrollment = pd.DataFrame([{
    "student_id": "S007",
    "subject": "IT102",
    "units": 3
}])

print("New enrollment event received:")
display(new_enrollment)

# Validate against student master
event = new_enrollment.merge(
    students,
    on="student_id",
    how="left",
    indicator=True
)

if event["_merge"].iloc[0] != "both":
    print("REJECTED: Student does not exist.")
elif event["units"].iloc[0] <= 0:
    print("REJECTED: Invalid units.")
else:
    print("ACCEPTED: Event passed validation.")


New enrollment event received:


,student_id,subject,units
0,S007,IT102,3


ACCEPTED: Event passed validation.


## 14. ETL Summary

### Extract
Get data from source systems.

### Integrate
Combine related datasets.

### Cleanse
Correct, remove, or handle poor-quality data.

### Transform
Convert data into the required structure and format.

### Validate
Check business and data-quality rules.

### Load
Store the processed data in the target system.

### Batch vs. Real-Time

| Batch | Real-Time |
|---|---|
| Processes data in groups | Processes data as it arrives |
| Scheduled | Continuous / event-driven |
| Daily payroll | Online payment |
| Nightly reporting | IoT sensor monitoring |
| Bulk import | Live transaction |

**Key takeaway:** ETL is not simply moving data. It is the controlled process of extracting, integrating, improving, transforming, validating, and loading data for reliable use.
